# Baseline MobileCount trên ShanghaiTech

Notebook train và đánh giá **MobileCount** làm mốc cho notebook cải tiến `improved_mobilecount`. Hai notebook dùng **chung** cách chia dữ liệu, density map, augmentation cơ bản, quy trình train (lr cosine, cùng số epoch, cùng seed) và cách đánh giá, nên so sánh giữa chúng là công bằng.

**MobileCount** là model encoder-decoder nhẹ, nhắm tới chạy thời gian thực:

- **Encoder MobileNetV2** pretrained ImageNet (depthwise separable conv, ít phép tính), lấy đặc trưng ở 4 mức 1/4, 1/8, 1/16, 1/32.
- **Decoder Light-Weight RefineNet:** từ mức sâu nhất, mỗi bước chiếu về 64 kênh → **Chained Residual Pooling** (gom ngữ cảnh rộng với rất ít tham số) → phóng to, cộng với mức nông hơn. Density đầu ra bằng **1/4** ảnh vào.
- Dùng MobileNetV2 chuẩn của torchvision; toàn model khoảng 2 triệu tham số.

**Chuẩn bị trên Kaggle**

1. **Add Input** → dataset `tthien/shanghaitech`. Bật **GPU** và **Internet** (tải trọng số MobileNetV2).
2. Đặt `QUICK = True` → **Run All** để kiểm tra trong vài phút; sau đó `QUICK = False` → **Save Version → Save & Run All**. Khoảng 45 phút trên T4.
3. Sau khi chạy xong, **Add Input output của notebook này vào notebook cải tiến**; notebook đó sẽ dùng lại kết quả baseline thay vì train lại.

**Nội dung:** 0. Cấu hình → 1. Dữ liệu → 2. Density map → 3. Dataset → 4. Mô hình (tham số, FLOPs, FPS) → 5. Công cụ train và đánh giá → 6. Train và kết quả → 7. Tổng kết.

In [ ]:
import contextlib, glob, io, json, math, os, random, re, shutil, time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from IPython.display import display
from PIL import Image, ImageFilter
from scipy.spatial import cKDTree
from torch.utils.data import DataLoader, Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| torchvision', torchvision.__version__)
if device.type == 'cuda':
    prop = torch.cuda.get_device_properties(0)
    GPU_NAME = prop.name
    print(f'GPU: {prop.name}, {prop.total_memory / 1024**3:.1f} GB')
else:
    GPU_NAME = 'CPU'
    print('CẢNH BÁO: không có GPU. Vào Settings → Accelerator → chọn GPU.')

plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})
NOTEBOOK_START = time.time()

## 0. Cấu hình

In [ ]:
PART = 'B'        # 'B': đường phố, mật độ vừa (gần camera demo) | 'A': ảnh Internet, rất đông
QUICK = False     # True: chạy thử cả notebook trong vài phút (8 ảnh mỗi tập, 2 epoch, 1 seed)

DATASET_DIR = '/kaggle/input/datasets/tthien/shanghaitech'
WORK_DIR = '/kaggle/working'
INPUT_DIR = '/kaggle/input'


def find_data_root(*bases):
    """Thư mục chứa part_A/part_B (dataset có thể lồng thêm một cấp ShanghaiTech/)."""
    for base in bases:
        for dirpath, dirnames, _ in os.walk(base):
            if 'part_A' in dirnames or 'part_B' in dirnames:
                return dirpath
    raise FileNotFoundError(f'Không thấy thư mục part_A/part_B trong {bases}. Kiểm tra lại Add Input.')


DATA_ROOT = find_data_root(DATASET_DIR, INPUT_DIR)
print('Dữ liệu:', DATA_ROOT)

# ---- Quy trình train, GIỐNG NHAU ở notebook baseline và notebook cải tiến ----
SEEDS = [42]               # 1 seed cho nhanh; thêm 43, 44 nếu cần trung bình ± độ lệch chuẩn
EPOCHS = 400               # mọi phiên bản train đủ 400 epoch, không dừng sớm; lr cosine về LR_MIN ở epoch cuối
SCHEDULE = 'cosine400'     # ghi vào kết quả để không dùng lại lần train theo cấu hình khác
TRAIN_PROBE = 40           # số ảnh train cố định để đo train MAE, so với val MAE (phát hiện overfit)
NOISE_REL = 0.10           # chỉ 1 seed: chênh lệch dưới 10% coi là dao động giữa các lần train
MAX_HOURS_PER_RUN = 1.5    # giới hạn an toàn cho mỗi lần train (một phiên bản, một seed)
LR, LR_MIN = 1e-4, 1e-6    # learning rate giảm theo cosine từ LR xuống LR_MIN
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 8
CROP = 512 if PART == 'B' else 256
STRIDE = 4                 # density của MobileCount bằng 1/4 ảnh vào
VAL_RATIO = 0.1
VAL_EVERY = 5
DENSITY_SCALE = 100.0
FIXED_SIGMA = 15
ADAPTIVE_BETA = 0.3
MASK_THRESH = 1e-6         # ô có density GT lớn hơn ngưỡng này = "có người" (phủ hết Gaussian, tới 3 sigma)
NUM_WORKERS = 4
N_SAMPLES = 6
FPS_SIZE = (768, 1024)
DATA_LIMIT = None

REUSE_PREVIOUS = True      # dùng lại kết quả đã có trong INPUT_DIR thay vì train lại

VARIANTS = {'baseline': dict(label='MobileCount (baseline)', backbone='v2', mask=False, aspp=False, aug=False, kd=False)}
RUN_VARIANTS = ['baseline']
PAPER = {}

if QUICK:
    DATA_LIMIT, EPOCHS, SEEDS, VAL_EVERY, N_SAMPLES, CROP, MAX_HOURS_PER_RUN = 8, 2, [42], 1, 2, 256, 0.1

SIGMA_MODE = 'adaptive' if PART == 'A' else 'fixed'
os.makedirs(WORK_DIR, exist_ok=True)
assert os.path.isdir(os.path.join(DATA_ROOT, f'part_{PART}')), f'Không có part_{PART} trong {DATA_ROOT}'
assert CROP % 32 == 0


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


seed_everything(SEEDS[0])
display(pd.DataFrame(VARIANTS).T.loc[RUN_VARIANTS])
print(f'Part {PART} | {EPOCHS} epoch, lr cosine {LR} → {LR_MIN} | patch {CROP} | batch {BATCH_SIZE} | '
      f'seed {SEEDS} → {len(RUN_VARIANTS) * len(SEEDS)} lần train' + (' | CHẾ ĐỘ CHẠY THỬ' if QUICK else ''))

## 1. Dữ liệu ShanghaiTech

Mỗi ảnh có một file `GT_IMG_x.mat` chứa tọa độ `(x, y)` của từng đầu người. Tập train gốc được tách 10% làm validation (chọn checkpoint); tập test chỉ dùng một lần ở cuối. Cách chia cố định (seed 42) và giống hệt nhau ở cả hai notebook.

In [ ]:
def natural_key(path):
    return [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', os.path.basename(path))]


def gt_path(img_path):
    split_dir = os.path.dirname(os.path.dirname(img_path))
    name = 'GT_' + os.path.splitext(os.path.basename(img_path))[0] + '.mat'
    for folder in ('ground-truth', 'ground_truth', 'gt'):
        path = os.path.join(split_dir, folder, name)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f'Không thấy nhãn cho {img_path}')


def load_points(img_path):
    info = sio.loadmat(gt_path(img_path))['image_info']
    return np.asarray(info[0, 0][0, 0][0], dtype=np.float32).reshape(-1, 2)


def load_split(split):
    pattern = os.path.join(DATA_ROOT, f'part_{PART}', split, 'images', '*.jpg')
    recs = []
    for path in sorted(glob.glob(pattern), key=natural_key)[:DATA_LIMIT]:
        w, h = Image.open(path).size
        recs.append({'name': os.path.basename(path), 'path': path, 'points': load_points(path), 'h': h, 'w': w})
    return recs


def describe(recs):
    c = np.array([len(r['points']) for r in recs])
    return {'số ảnh': len(recs), 'tổng số người': int(c.sum()), 'ít nhất': int(c.min()), 'nhiều nhất': int(c.max()),
            'trung bình': round(c.mean(), 1), 'trung vị': float(np.median(c)), 'độ lệch chuẩn': round(c.std(), 1)}


t0 = time.time()
train_all, test_recs = load_split('train_data'), load_split('test_data')
test_by_name = {r['name']: r for r in test_recs}
order = np.random.RandomState(42).permutation(len(train_all))
n_val = max(1, round(len(train_all) * VAL_RATIO))
val_recs = [train_all[i] for i in sorted(order[:n_val])]
train_recs = [train_all[i] for i in sorted(order[n_val:])]
print(f'Đọc {len(train_all) + len(test_recs)} ảnh và nhãn trong {time.time() - t0:.1f}s')
display(pd.DataFrame({'train': describe(train_recs), 'val': describe(val_recs), 'test': describe(test_recs)}).T)
sizes = pd.Series([f"{r['w']}×{r['h']}" for r in train_all + test_recs]).value_counts()
print(f'Số kích thước ảnh khác nhau: {len(sizes)}; phổ biến nhất: {sizes.index[0]} ({sizes.iloc[0]} ảnh)')

In [ ]:
counts = {name: np.array([len(r['points']) for r in recs])
          for name, recs in (('train', train_recs), ('val', val_recs), ('test', test_recs))}
fig, axes = plt.subplots(1, 2, figsize=(16, 4.2))
bins = np.linspace(0, max(c.max() for c in counts.values()), 30)
for name, c in counts.items():
    axes[0].hist(c, bins=bins, alpha=0.55, label=f'{name} ({len(c)} ảnh)')
axes[0].set(title=f'Phân bố số người mỗi ảnh (Part {PART})', xlabel='số người', ylabel='số ảnh')
axes[0].legend()
axes[1].boxplot(list(counts.values()))
axes[1].set_xticks(range(1, len(counts) + 1), list(counts.keys()))
axes[1].set(title='Số người mỗi ảnh theo tập', ylabel='số người')
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR, f'data_stats_part{PART}.png'), bbox_inches='tight'); plt.show()

picks = random.Random(0).sample(train_recs, min(4, len(train_recs)))
fig, axes = plt.subplots(1, len(picks), figsize=(5 * len(picks), 4.2), squeeze=False)
for ax, r in zip(axes[0], picks):
    ax.imshow(Image.open(r['path']).convert('RGB'))
    ax.scatter(r['points'][:, 0], r['points'][:, 1], s=4, c='red')
    ax.set_title(f"{r['name']}: {len(r['points'])} người"); ax.axis('off')
plt.suptitle('Ảnh mẫu với nhãn điểm đầu người (chấm đỏ)'); plt.tight_layout(); plt.show()

## 2. Density map ground truth và vùng "có người"

- **Density map:** một Gaussian tại mỗi đầu người, cắt ở bán kính 3 sigma và chuẩn hóa để tổng bằng 1 cho mỗi người (Part B: sigma 15 px; Part A: sigma thích nghi). Khi train, density được cộng gộp xuống 1/4 cho khớp đầu ra của MobileCount.
- **Vùng "có người":** các ô density 1/4 lớn hơn `MASK_THRESH`, tức nằm trong bán kính 3 sigma của ít nhất một đầu người. Phần còn lại là **nền**: ở đó density ground truth bằng đúng 0.

Vùng này dùng để đo **đếm nhầm ở nền** (tổng density dự đoán rơi vào nền, tức chỗ cách mọi đầu người hơn 3 sigma), và làm nhãn cho nhánh mask C1 ở notebook cải tiến. Ngưỡng được chọn để phủ trọn Gaussian: nếu ngưỡng cao hơn, phần đuôi Gaussian rơi ra "nền", khiến cả model hoàn hảo cũng bị tính là đếm nhầm.

In [ ]:
_KERNELS = {}


def gaussian_kernel(sigma):
    key = round(float(sigma), 1)
    if key not in _KERNELS:
        r = max(1, int(math.ceil(3 * key)))
        ax = np.arange(-r, r + 1, dtype=np.float32)
        g = np.exp(-ax ** 2 / (2 * key ** 2))
        k = np.outer(g, g)
        _KERNELS[key] = (k / k.sum()).astype(np.float32)
    return _KERNELS[key]


def point_sigmas(points):
    if SIGMA_MODE == 'fixed' or len(points) < 4:
        return np.full(len(points), FIXED_SIGMA, np.float32)
    dist, _ = cKDTree(points).query(points, k=4)
    return np.clip(ADAPTIVE_BETA * dist[:, 1:].mean(axis=1), 1.0, 3 * FIXED_SIGMA)


def make_density(points, h, w):
    """Mỗi người là một Gaussian có tổng bằng 1, nên tổng density = số người."""
    den = np.zeros((h, w), np.float32)
    for (x, y), s in zip(points, point_sigmas(points)):
        x, y = min(max(int(x), 0), w - 1), min(max(int(y), 0), h - 1)
        k = gaussian_kernel(s)
        r = k.shape[0] // 2
        y0, y1, x0, x1 = max(y - r, 0), min(y + r + 1, h), max(x - r, 0), min(x + r + 1, w)
        patch = k[y0 - y + r:y1 - y + r, x0 - x + r:x1 - x + r]
        den[y0:y1, x0:x1] += patch / patch.sum()
    return den


def sum_pool(den, s):
    """Thu nhỏ density s lần mỗi chiều bằng cách cộng từng ô s×s (giữ nguyên tổng)."""
    h, w = den.shape
    H, W = math.ceil(h / s) * s, math.ceil(w / s) * s
    den = np.pad(den, ((0, H - h), (0, W - w)))
    return den.reshape(H // s, s, W // s, s).sum(axis=(1, 3))


t0 = time.time()
densities = {r['name']: make_density(r['points'], r['h'], r['w']) for r in train_all}
GT_TEST = {r['name']: sum_pool(make_density(r['points'], r['h'], r['w']), STRIDE) for r in test_recs}
print(f'Tạo density cho {len(densities)} ảnh train và {len(GT_TEST)} ảnh test trong {time.time() - t0:.1f}s')
print(f"Sai lệch lớn nhất giữa tổng density và số nhãn: "
      f"{max(abs(densities[r['name']].sum() - len(r['points'])) for r in train_all):.4f} người")
cover = [(g > MASK_THRESH).mean() for g in GT_TEST.values()]
leak = [g[g <= MASK_THRESH].sum() for g in GT_TEST.values()]
print(f'Vùng "có người" chiếm trung bình {np.mean(cover):.1%} diện tích ảnh test (ít nhất {np.min(cover):.1%}, '
      f'nhiều nhất {np.max(cover):.1%}); density GT nằm ngoài vùng này: {max(leak):.6f} người (phải bằng 0)')

In [ ]:
picks = sorted(train_recs, key=lambda r: len(r['points']))
picks = [picks[0], picks[len(picks) // 2], picks[-1]]
fig, axes = plt.subplots(len(picks), 3, figsize=(16, 4.3 * len(picks)), squeeze=False)
for row, r in zip(axes, picks):
    img = Image.open(r['path']).convert('RGB')
    small = sum_pool(densities[r['name']], STRIDE)
    mask = small > MASK_THRESH
    extent = (0, small.shape[1] * STRIDE, small.shape[0] * STRIDE, 0)
    row[0].imshow(img); row[0].scatter(r['points'][:, 0], r['points'][:, 1], s=3, c='red')
    row[0].set_title(f"{r['name']}: {len(r['points'])} người")
    row[1].imshow(small, cmap='jet'); row[1].set_title(f'Density 1/4 (tổng {small.sum():.1f})')
    row[2].imshow(img); row[2].imshow(mask, cmap='Greens', alpha=0.45, extent=extent)
    row[2].contour(mask, levels=[0.5], colors='lime', linewidths=1.5, extent=extent, origin='upper')
    row[2].set(xlim=(0, img.size[0]), ylim=(img.size[1], 0))
    row[2].set_title(f'Vùng "có người" ({mask.mean():.0%} diện tích); ngoài viền = nền')
    for ax in row:
        ax.axis('off')
plt.suptitle('Ảnh thưa nhất, trung bình và đông nhất trong tập train'); plt.tight_layout(); plt.show()

## 3. Dataset

Mỗi lần lấy một ảnh: cắt ngẫu nhiên một patch, lật ngang với xác suất 50%, chuẩn hóa màu theo ImageNet. Ảnh nhỏ hơn patch được đệm thêm số 0. Mỗi epoch dùng mỗi ảnh train đúng một lần.

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)


class CrowdPatches(Dataset):
    """Trả về (ảnh, density 1/4 đã nhân DENSITY_SCALE, vùng "có người" 1/4)."""

    def __init__(self, recs, densities, aug=False):
        self.recs, self.densities, self.aug = recs, densities, aug

    def __len__(self):
        return len(self.recs)

    def __getitem__(self, i):
        r = self.recs[i]
        img, den = Image.open(r['path']).convert('RGB'), self.densities[r['name']]
        arr = np.asarray(img, np.float32) / 255.0
        h, w = den.shape
        y, x = random.randint(0, max(h - CROP, 0)), random.randint(0, max(w - CROP, 0))
        arr, den = arr[y:y + CROP, x:x + CROP], den[y:y + CROP, x:x + CROP]
        arr = (arr - MEAN) / STD
        ph, pw = CROP - den.shape[0], CROP - den.shape[1]
        if ph or pw:
            arr = np.pad(arr, ((0, ph), (0, pw), (0, 0)))
            den = np.pad(den, ((0, ph), (0, pw)))
        if random.random() < 0.5:
            arr, den = arr[:, ::-1], den[:, ::-1]
        small = sum_pool(den, STRIDE)
        return (torch.from_numpy(np.ascontiguousarray(arr.transpose(2, 0, 1), dtype=np.float32)),
                torch.from_numpy(np.ascontiguousarray(small[None] * DENSITY_SCALE, dtype=np.float32)),
                torch.from_numpy(np.ascontiguousarray((small > MASK_THRESH)[None], dtype=np.float32)))


def make_loader(aug=False):
    return DataLoader(CrowdPatches(train_recs, densities, aug), batch_size=BATCH_SIZE, shuffle=True,
                      drop_last=len(train_recs) >= BATCH_SIZE, num_workers=NUM_WORKERS,
                      pin_memory=device.type == 'cuda', persistent_workers=NUM_WORKERS > 0)


def denormalize(chw):
    return np.clip(chw.transpose(1, 2, 0) * STD + MEAN, 0, 1)

In [ ]:
seed_everything(SEEDS[0])
ds = CrowdPatches(train_recs, densities)
n_show = min(4, len(train_recs))
fig, axes = plt.subplots(2, n_show, figsize=(4.5 * n_show, 8.5), squeeze=False)
for j in range(n_show):
    img, den, _ = ds[j]
    axes[0, j].imshow(denormalize(img.numpy())); axes[0, j].set_title(f'patch {CROP}×{CROP}')
    axes[1, j].imshow(den[0].numpy(), cmap='jet')
    axes[1, j].set_title(f'density {den.shape[-1]}×{den.shape[-2]}: {den.sum() / DENSITY_SCALE:.1f} người')
    for ax in axes[:, j]:
        ax.axis('off')
plt.suptitle('Patch train sau khi cắt ngẫu nhiên và lật ngang'); plt.tight_layout(); plt.show()

## 4. Mô hình MobileCount

```
Ảnh → MobileNetV2 (1/4, 1/8, 1/16, 1/32)
        1/32 → conv 1×1 → CRP ─┐
        1/16 → conv 1×1 ─── + ← phóng to → CRP ─┐
        1/8  → conv 1×1 ─── + ← phóng to → CRP ─┐
        1/4  → conv 1×1 ─── + ← phóng to → CRP → head → density 1/4
```

In [ ]:
class ChainedResidualPool(nn.Module):
    """Chuỗi max pool 5×5 + conv 1×1 cộng dồn: gom ngữ cảnh rộng với rất ít tham số (Light-Weight RefineNet)."""

    def __init__(self, channels, n_stages=4):
        super().__init__()
        self.pool = nn.MaxPool2d(5, stride=1, padding=2)
        self.convs = nn.ModuleList([nn.Conv2d(channels, channels, 1, bias=False) for _ in range(n_stages)])

    def forward(self, x):
        top = x
        for conv in self.convs:
            top = conv(self.pool(top))
            x = x + top
        return x


class MobileCount(nn.Module):
    """MobileCount: encoder MobileNetV2 + decoder Light-Weight RefineNet."""

    def __init__(self, pretrained=True, channels=64):
        super().__init__()
        weights = torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
        features = torchvision.models.mobilenet_v2(weights=weights).features
        # Các mức đặc trưng: 1/4 (24 kênh), 1/8 (32), 1/16 (96), 1/32 (320)
        self.encoder = nn.ModuleList([features[:4], features[4:7], features[7:14], features[14:18]])
        self.project = nn.ModuleList([nn.Conv2d(c, channels, 1, bias=False) for c in (24, 32, 96)])
        self.deep = nn.Sequential(nn.Conv2d(320, channels, 1, bias=False), nn.ReLU(inplace=True))
        self.crp = nn.ModuleList([ChainedResidualPool(channels) for _ in range(4)])
        self.fuse = nn.ModuleList([nn.Conv2d(channels, channels, 1, bias=False) for _ in range(3)])
        self.head = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True),
                                  nn.Conv2d(channels, 1, 1))

    def forward(self, x, with_mask=False):
        feats = []
        for stage in self.encoder:
            x = stage(x)
            feats.append(x)
        y = self.crp[3](self.deep(feats[3]))
        for i in (2, 1, 0):
            y = F.interpolate(self.fuse[i](y), size=feats[i].shape[-2:], mode='bilinear', align_corners=False)
            y = self.crp[i](F.relu(self.project[i](feats[i]) + y))
        den = self.head(y)
        return (den, None) if with_mask else den


def build_variant(key, pretrained=True):
    return MobileCount(pretrained=pretrained).to(device)

In [ ]:
def sync():
    if device.type == 'cuda':
        torch.cuda.synchronize()


@torch.no_grad()
def measure_fps(model, size, half=False, runs=30):
    x = torch.randn(1, 3, *size, device=device)
    ctx = torch.autocast('cuda', dtype=torch.float16) if half and device.type == 'cuda' else contextlib.nullcontext()
    with ctx:
        for _ in range(5):
            model(x)
        sync(); t = time.perf_counter()
        for _ in range(runs):
            model(x)
        sync()
    return runs / (time.perf_counter() - t)


@torch.no_grad()
def count_gmacs(model, size):
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with FlopCounterMode(display=False) as counter:
            model(torch.zeros(1, 3, *size, device=device))
        return counter.get_total_flops() / 2e9
    except Exception as e:
        print('  không đo được FLOPs:', e)
        return None


def model_stats(model):
    model.eval()
    return {'params_M': sum(p.numel() for p in model.parameters()) / 1e6, 'GMACs': count_gmacs(model, FPS_SIZE),
            'FPS': measure_fps(model, FPS_SIZE), 'FPS_fp16': measure_fps(model, FPS_SIZE, half=True)}


MODEL_STATS = {}
for key in RUN_VARIANTS:
    model = build_variant(key).eval()
    with torch.no_grad():
        out = model(torch.zeros(1, 3, CROP, CROP, device=device))
    assert out.shape[-2:] == (CROP // STRIDE, CROP // STRIDE), f'{key}: kích thước đầu ra sai {tuple(out.shape)}'
    MODEL_STATS[key] = model_stats(model)
    if key == RUN_VARIANTS[0]:
        blocks = pd.Series({n: sum(p.numel() for p in m.parameters()) for n, m in model.named_children()}, name='tham số')
        print(f'Số tham số từng khối của {VARIANTS[key]["label"]}:')
        display(blocks.to_frame().assign(tỉ_lệ=(blocks / blocks.sum()).map('{:.1%}'.format)))
    del model
if device.type == 'cuda':
    torch.cuda.empty_cache()

stats = pd.DataFrame(MODEL_STATS).T
stats.insert(0, 'phiên bản', [VARIANTS[k]['label'] for k in stats.index])
print(f'Ảnh {FPS_SIZE[1]}×{FPS_SIZE[0]}, batch 1, {GPU_NAME}. FPS đo ở fp32 và fp16 (mixed precision, cách chạy khi triển khai).')
display(stats.style.format({'params_M': '{:.3f}', 'GMACs': '{:.2f}', 'FPS': '{:.1f}', 'FPS_fp16': '{:.1f}'}, na_rep='–'))

## 5. Công cụ train và đánh giá

Giống hệt nhau ở notebook baseline và notebook cải tiến.

- **Train:** Adam, mixed precision. Mỗi `VAL_EVERY` epoch đo MAE trên validation; checkpoint có **val MAE thấp nhất** là kết quả chính. Tập test không tham gia vào việc chọn checkpoint.
- **Learning rate cosine, không dừng sớm:** lr giảm dần theo cosine từ `LR` xuống `LR_MIN`, mọi phiên bản train đủ `EPOCHS` epoch. Cùng thời lượng train thì so sánh giữa các phiên bản mới công bằng.
- **Hai checkpoint:** checkpoint val MAE tốt nhất (kết quả chính) và checkpoint epoch cuối đều được đánh giá trên test. Hai số gần nhau nghĩa là việc chọn theo tập val nhỏ (40 ảnh) không làm kết quả bị may rủi.
- **Theo dõi overfit:** mỗi lần đo val, notebook đo thêm MAE trên `TRAIN_PROBE` ảnh train cố định. Biểu đồ vẽ train MAE (nét đứt) cạnh val MAE: khoảng cách giữa hai đường tăng dần là dấu hiệu overfit.
- **Số seed:** mặc định 1 seed (`SEEDS = [42]`). Kết quả một lần train có dao động ngẫu nhiên, nên khi so sánh, chênh lệch dưới `NOISE_REL` (10%) được coi là dao động. Thêm seed 43, 44 thì kết quả báo cáo dạng trung bình ± độ lệch chuẩn.
- **Đánh giá test** trên ảnh kích thước gốc:
  - **MAE, MSE** (theo quy ước của lĩnh vực đếm người, MSE thực chất là căn bậc hai sai số bình phương trung bình).
  - **GAME(L):** chia ảnh thành lưới 2^L × 2^L ô, cộng sai số đếm từng ô; đo cả việc đếm đúng chỗ.
  - **Đếm nhầm ở nền:** tổng density dự đoán ở các ô cách mọi đầu người hơn 3 sigma.
  - **IoU mask** (chỉ với bản có C1): độ trùng giữa vùng đám đông dự đoán và thực tế.
- **Chạy tiếp khi bị ngắt:** mỗi lần train (một phiên bản, một seed) lưu kết quả riêng. Add Input output của version trước thì những lần đã xong được dùng lại.

In [ ]:
BINS = {'A': [0, 250, 500, 1000, np.inf], 'B': [0, 50, 100, 200, np.inf]}
BIN_LABELS = [f'{int(a) + 1 if a else 0}–{int(b)}' if np.isfinite(b) else f'>{int(a)}'
              for a, b in zip(BINS[PART][:-1], BINS[PART][1:])]
METRICS = ['MAE', 'MSE', 'GAME1', 'GAME2', 'GAME3', 'Đếm nhầm ở nền', 'IoU mask']


def density_bin(gt):
    return pd.cut(gt, BINS[PART], labels=BIN_LABELS, include_lowest=True)


@torch.no_grad()
def predict(model, path, stride=STRIDE, with_mask=False):
    """Density dự đoán (đã chia DENSITY_SCALE) và mask xác suất (nếu có) cho cả ảnh."""
    img = (np.asarray(Image.open(path).convert('RGB'), np.float32) / 255.0 - MEAN) / STD
    h, w, _ = img.shape
    H, W = math.ceil(h / 32) * 32, math.ceil(w / 32) * 32
    img = np.pad(img, ((0, H - h), (0, W - w), (0, 0)))
    x = torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1)))[None].to(device)
    hh, ww = math.ceil(h / stride), math.ceil(w / stride)
    if with_mask:
        den, logit = model(x, with_mask=True)
        mask = None if logit is None else torch.sigmoid(logit)[0, 0].float().cpu().numpy()[:hh, :ww]
    else:
        den, mask = model(x), None
    return den[0, 0].float().cpu().numpy()[:hh, :ww] / DENSITY_SCALE, mask


def evaluate_counts(model, recs):
    was_training = model.training
    model.eval()
    err = np.array([predict(model, r['path'])[0].sum() - len(r['points']) for r in recs])
    model.train(was_training)
    return {'MAE': float(np.abs(err).mean()), 'MSE': float(np.sqrt((err ** 2).mean()))}


def game(pred, gt, level):
    n = 2 ** level
    return sum(abs(p2.sum() - g2.sum())
               for p1, g1 in zip(np.array_split(pred, n, 0), np.array_split(gt, n, 0))
               for p2, g2 in zip(np.array_split(p1, n, 1), np.array_split(g1, n, 1)))


def evaluate_full(model, recs, stride=STRIDE, with_mask=False):
    model.eval()
    rows = []
    for r in recs:
        gt = GT_TEST[r['name']] if stride == STRIDE else sum_pool(GT_TEST[r['name']], stride // STRIDE)
        pred, mask = predict(model, r['path'], stride, with_mask)
        background = gt <= MASK_THRESH
        row = {'name': r['name'], 'gt': len(r['points']), 'pred': float(pred.sum()),
               'bg_count': float(np.clip(pred, 0, None)[background].sum())}
        for level in (1, 2, 3):
            row[f'GAME{level}'] = float(game(pred, gt, level))
        if mask is not None:
            m, g = mask > 0.5, ~background
            row['mask_iou'] = float((m & g).sum() / max((m | g).sum(), 1))
        rows.append(row)
    df = pd.DataFrame(rows)
    df['error'] = df['pred'] - df['gt']
    return df


def summarize(df):
    out = {'MAE': df['error'].abs().mean(), 'MSE': np.sqrt((df['error'] ** 2).mean()),
           **{f'GAME{l}': df[f'GAME{l}'].mean() for l in (1, 2, 3)}, 'Đếm nhầm ở nền': df['bg_count'].mean()}
    if 'mask_iou' in df:
        out['IoU mask'] = df['mask_iou'].mean()
    return {k: float(v) for k, v in out.items()}


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler('cuda', enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def files_of(key, seed):
    tag = f'mc_{key}_s{seed}_part{PART}'
    return {'results': f'results_{tag}.json', 'preds': f'preds_{tag}.csv', 'history': f'history_{tag}.csv',
            'samples': f'samples_{tag}.npz', 'weights': f'best_{tag}.pth', 'last': f'last_{tag}.pth'}


def load_record(key, seed, folder=WORK_DIR):
    f = {k: os.path.join(folder, v) for k, v in files_of(key, seed).items()}
    with open(f['results'], encoding='utf-8') as fh:
        res = json.load(fh)
    with np.load(f['samples']) as s:
        samples = {k: s[k] for k in s.files}
    return {'res': res, 'preds': pd.read_csv(f['preds']), 'history': pd.read_csv(f['history']),
            'samples': samples, 'weights': f['weights']}


def find_previous(key, seed):
    if not REUSE_PREVIOUS or not os.path.isdir(INPUT_DIR):
        return None
    for path in glob.glob(os.path.join(INPUT_DIR, '**', files_of(key, seed)['results']), recursive=True):
        with open(path, encoding='utf-8') as fh:
            res = json.load(fh)
        if not res.get('quick_test') and res.get('config', {}).get('schedule') == SCHEDULE:
            return os.path.dirname(path)
    return None


def mean_std(key, metric):
    vals = [r['res'][metric] for r in RESULTS.get(key, []) if metric in r['res']]
    if not vals:
        return np.nan, np.nan
    return float(np.mean(vals)), float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0


def fmt_ms(mu, sd):
    if np.isnan(mu):
        return '–'
    return f'{mu:.2f} ± {sd:.2f}' if len(SEEDS) > 1 else f'{mu:.2f}'


def mean_preds(key):
    """Dự đoán trung bình qua các seed cho từng ảnh test."""
    df = pd.concat([r['preds'] for r in RESULTS[key]])
    cols = [c for c in ('pred', 'bg_count', 'mask_iou') if c in df]
    out = df.groupby('name').agg({'gt': 'first', **{c: 'mean' for c in cols}})
    out['error'] = out['pred'] - out['gt']
    return out


SAMPLE_NAMES = [test_recs[i]['name'] for i in np.linspace(0, len(test_recs) - 1, N_SAMPLES).astype(int)]
RESULTS = {}

In [ ]:
train_probe = random.Random(0).sample(train_recs, min(TRAIN_PROBE, len(train_recs)))


def train_once(key, seed):
    cfg, files = VARIANTS[key], {k: os.path.join(WORK_DIR, v) for k, v in files_of(key, seed).items()}
    prev = find_previous(key, seed)
    if prev:
        for fname in files_of(key, seed).values():
            if os.path.exists(os.path.join(prev, fname)):
                shutil.copy(os.path.join(prev, fname), WORK_DIR)
        rec = load_record(key, seed)
        print(f"  seed {seed}: dùng lại kết quả từ {prev} (MAE {rec['res']['MAE']:.2f}), không train lại.")
        return rec

    seed_everything(seed)
    model = build_variant(key)
    loader = make_loader(cfg['aug'])
    use_teacher = teacher if cfg['kd'] else None
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
    use_amp = device.type == 'cuda'
    scaler = make_scaler(use_amp)

    history, best_mae, best_epoch = [], float('inf'), 0
    start = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        t_epoch, parts = time.time(), {'loss': [], 'loss_density': [], 'loss_mask': [], 'loss_kd': []}
        for img, den, mask_gt in loader:
            img, den, mask_gt = (t.to(device, non_blocking=True) for t in (img, den, mask_gt))
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                out, logit = model(img, with_mask=True)
                target = use_teacher(img) if use_teacher is not None else None
            out = out.float()
            loss = F.mse_loss(out, den)
            parts['loss_density'].append(loss.item())
            if logit is not None:
                l_mask = F.binary_cross_entropy_with_logits(logit.float(), mask_gt)
                loss = loss + MASK_LAMBDA * l_mask
                parts['loss_mask'].append(l_mask.item())
            if target is not None:
                l_kd = F.mse_loss(F.avg_pool2d(out, 2) * 4, target.float())
                loss = loss + KD_ALPHA * l_kd
                parts['loss_kd'].append(l_kd.item())
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            parts['loss'].append(loss.item())
        row = {'epoch': epoch, 'lr': optimizer.param_groups[0]['lr'], 'epoch_s': time.time() - t_epoch,
               **{k: float(np.mean(v)) if v else np.nan for k, v in parts.items()}}
        scheduler.step()
        time_over = (time.time() - start) / 3600 > MAX_HOURS_PER_RUN
        if epoch % VAL_EVERY == 0 or epoch == EPOCHS or time_over:
            m, tm = evaluate_counts(model, val_recs), evaluate_counts(model, train_probe)
            row.update(val_MAE=m['MAE'], val_MSE=m['MSE'], train_MAE=tm['MAE'])
            note = ''
            if m['MAE'] < best_mae:
                best_mae, best_epoch = m['MAE'], epoch
                torch.save(model.state_dict(), files['weights'])
                note = '  ← tốt nhất'
            if epoch % (VAL_EVERY * 5) == 0 or note or epoch == EPOCHS or time_over:
                print(f"  seed {seed} | epoch {epoch:3d} | lr {row['lr']:.1e} | loss {row['loss']:.5f} | "
                      f"train MAE {tm['MAE']:6.2f} | val MAE {m['MAE']:6.2f} | val MSE {m['MSE']:6.2f} | {row['epoch_s']:4.1f}s/epoch | "
                      f"{(time.time() - start) / 60:5.1f} phút{note}")
        history.append(row)
        if time_over:
            print(f'  seed {seed}: dừng ở epoch {epoch} vì đã hết {MAX_HOURS_PER_RUN} giờ.')
            break
    train_minutes = (time.time() - start) / 60
    history = pd.DataFrame(history)
    history.to_csv(files['history'], index=False)

    # Checkpoint epoch cuối: đánh giá test để đối chiếu với checkpoint chọn theo val
    torch.save(model.state_dict(), files['last'])
    last = summarize(evaluate_full(model, test_recs, with_mask=cfg['mask']))

    model.load_state_dict(torch.load(files['weights'], map_location=device))
    test_df = evaluate_full(model, test_recs, with_mask=cfg['mask'])
    test_df.to_csv(files['preds'], index=False)
    samples = {}
    for i, n in enumerate(SAMPLE_NAMES):
        den, mask = predict(model, test_by_name[n]['path'], with_mask=cfg['mask'])
        samples[f'den_{i}'] = den.astype(np.float16)
        if mask is not None:
            samples[f'mask_{i}'] = mask.astype(np.float16)
    np.savez_compressed(files['samples'], names=np.array(SAMPLE_NAMES), **samples)

    res = {'variant': key, 'label': cfg['label'], 'seed': seed, 'part': PART, **summarize(test_df),
           **MODEL_STATS[key], 'device': GPU_NAME, 'epochs_run': int(history['epoch'].max()),
           'best_epoch': best_epoch, 'best_val_MAE': best_mae, 'train_minutes': train_minutes, 'quick_test': QUICK,
           'last_epoch': int(history['epoch'].max()), 'last_MAE': last['MAE'], 'last_MSE': last['MSE'],
           'config': {**cfg, 'epochs': EPOCHS, 'schedule': SCHEDULE, 'lr': LR, 'lr_min': LR_MIN, 'batch_size': BATCH_SIZE, 'crop': CROP,
                      'mask_thresh': MASK_THRESH}}
    with open(files['results'], 'w', encoding='utf-8') as fh:
        json.dump(res, fh, ensure_ascii=False, indent=2)
    print(f"  seed {seed} xong sau {train_minutes:.1f} phút: epoch tốt nhất {best_epoch} (val MAE {best_mae:.2f}) "
          f"→ test MAE {res['MAE']:.2f}, MSE {res['MSE']:.2f} | epoch cuối: test MAE {last['MAE']:.2f}, MSE {last['MSE']:.2f}\n")
    del model, optimizer, loader
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return load_record(key, seed)


def run_variant(key):
    cfg = VARIANTS[key]
    print(f"{cfg['label']}: {len(train_recs)} ảnh train, {EPOCHS} epoch × {len(SEEDS)} seed. Loss: density"
          + (' + mask' if cfg['mask'] else '') + (' + distillation' if cfg['kd'] else '')
          + (' | augmentation C5' if cfg['aug'] else '') + '\n')
    records = [train_once(key, seed) for seed in SEEDS]
    mae = [r['res']['MAE'] for r in records]
    print(f"{cfg['label']}: test MAE theo seed {[round(v, 2) for v in mae]} → {np.mean(mae):.2f} ± "
          f"{np.std(mae, ddof=1) if len(mae) > 1 else 0:.2f}")
    return records

In [ ]:
def show_examples(model, key, df, title):
    has_mask = VARIANTS[key]['mask']
    n_cols = 4 if has_mask else 3
    fig, axes = plt.subplots(len(df), n_cols, figsize=(5.2 * n_cols, 4.2 * len(df)), squeeze=False)
    for row, (name, item) in zip(axes, df.iterrows()):
        r = test_by_name[name]
        img = Image.open(r['path']).convert('RGB')
        w, h = img.size
        gt = GT_TEST[name]
        pred, mask = predict(model, r['path'], with_mask=has_mask)
        vmax = max(gt.max(), pred.max())
        extent = (0, pred.shape[1] * STRIDE, pred.shape[0] * STRIDE, 0)
        row[0].imshow(img); row[0].set_title(f"{name}: thật {int(item['gt'])}, dự đoán TB {item['pred']:.1f}")
        row[1].imshow(gt, cmap='jet', vmin=0, vmax=vmax); row[1].set_title(f'Density ground truth ({gt.sum():.1f})')
        row[2].imshow(img); row[2].imshow(pred, cmap='jet', alpha=0.5, vmin=0, vmax=vmax, extent=extent)
        row[2].set(xlim=(0, w), ylim=(h, 0))
        row[2].set_title(f'Dự đoán seed {SEEDS[0]} ({pred.sum():.1f})')
        if has_mask:
            row[3].imshow(img)
            row[3].imshow(mask, cmap='Greens', alpha=0.45, vmin=0, vmax=1, extent=extent)
            row[3].contour(mask, levels=[0.5], colors='lime', linewidths=1.5, extent=extent, origin='upper')
            row[3].contour(gt > MASK_THRESH, levels=[0.5], colors='red', linewidths=1, linestyles='--',
                           extent=extent, origin='upper')
            row[3].set(xlim=(0, w), ylim=(h, 0))
            row[3].set_title('Khoanh vùng: dự đoán (xanh), thật (đỏ)')
        for ax in row:
            ax.axis('off')
    plt.suptitle(title); plt.tight_layout(); plt.show()


def report(key):
    cfg, recs = VARIANTS[key], RESULTS[key]

    # Kết quả từng seed và trung bình
    rows = []
    for r in recs:
        res = r['res']
        rows.append({'seed': res['seed'], **{m: res.get(m, np.nan) for m in METRICS},
                     'MAE epoch cuối': res.get('last_MAE', np.nan), 'MSE epoch cuối': res.get('last_MSE', np.nan),
                     'epoch tốt nhất': res['best_epoch'], 'epoch đã chạy': res['epochs_run'],
                     'phút train': res['train_minutes']})
    per_seed = pd.DataFrame(rows).set_index('seed').dropna(axis=1, how='all')
    shown = [m for m in METRICS if m in per_seed]
    summary = pd.DataFrame({m: [fmt_ms(*mean_std(key, m))] for m in shown}, index=['trung bình ± độ lệch chuẩn' if len(SEEDS) > 1 else 'kết quả'])
    print(f"{cfg['label']}: kết quả trên {len(test_recs)} ảnh test. MAE, MSE, GAME, đếm nhầm ở nền: thấp hơn là tốt hơn.")
    two = shown + [c for c in ('MAE epoch cuối', 'MSE epoch cuối') if c in per_seed]
    display(per_seed.style.format('{:.2f}', subset=two).format('{:.0f}', subset=['phút train']))
    display(summary)

    ref = PAPER.get('MobileCount', {}).get(PART)
    compare = {}
    if ref:
        compare['MobileCount (paper)'] = {'MAE': f'{ref[0]:.2f}', 'MSE': f'{ref[1]:.2f}'}
    if key != 'baseline' and 'baseline' in RESULTS:
        compare['MobileCount baseline (notebook)'] = {m: fmt_ms(*mean_std('baseline', m)) for m in shown}
    if compare:
        compare = {cfg['label']: {m: summary.iloc[0][m] for m in shown}, **compare}
        print('So sánh:')
        display(pd.DataFrame(compare).T.fillna('–'))

    # Đường cong train
    fig, axes = plt.subplots(1, 3, figsize=(22, 4.2))
    for r in recs:
        h = r['history']
        v = h.dropna(subset=['val_MAE'])
        axes[0].plot(h['epoch'], h['loss_density'], label=f"seed {r['res']['seed']}")
        axes[1].plot(v['epoch'], v['val_MAE'], 'o-', ms=2, label=f"val, seed {r['res']['seed']}")
        if 'train_MAE' in v:
            axes[1].plot(v['epoch'], v['train_MAE'], '--', label=f"train ({TRAIN_PROBE} ảnh), seed {r['res']['seed']}")
        axes[1].axvline(r['res']['best_epoch'], color='gray', ls=':', lw=1)
    h0 = recs[0]['history']
    for c in ('loss_mask', 'loss_kd'):
        if c in h0 and h0[c].notna().any():
            axes[0].plot(h0['epoch'], h0[c], ls='--', label=f"{c.replace('loss_', '')} (seed {SEEDS[0]})")
    axes[0].set(title='Loss train', xlabel='epoch', yscale='log'); axes[0].legend(fontsize=8)
    axes[1].set(title='MAE train và val (chấm dọc: checkpoint được chọn)', xlabel='epoch', ylabel='MAE', yscale='log'); axes[1].legend(fontsize=8)
    axes[2].plot(h0['epoch'], h0['lr']); axes[2].set(title='Learning rate', xlabel='epoch', yscale='log')
    plt.suptitle(cfg['label']); plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR, f'curves_mc_{key}_part{PART}.png'), bbox_inches='tight'); plt.show()

    # Theo nhóm mật độ, trên dự đoán trung bình qua các seed
    df = mean_preds(key)
    df['nhóm mật độ'] = density_bin(df['gt'])
    by_bin = df.groupby('nhóm mật độ', observed=False).agg(
        số_ảnh=('gt', 'size'), MAE=('error', lambda e: e.abs().mean()),
        lệch_TB=('error', 'mean'), nhầm_ở_nền=('bg_count', 'mean'))
    print('Theo nhóm mật độ (dự đoán trung bình qua các seed; lệch dương = đếm thừa):')
    display(by_bin.round(2))

    fig, axes = plt.subplots(1, 3, figsize=(20, 4.5))
    lim = max(df['gt'].max(), df['pred'].max()) * 1.05
    axes[0].scatter(df['gt'], df['pred'], s=14, alpha=0.6)
    axes[0].plot([0, lim], [0, lim], 'r--')
    axes[0].set(title='Dự đoán (TB các seed) so với thực tế', xlabel='số người thật', ylabel='dự đoán',
                xlim=(0, lim), ylim=(0, lim))
    axes[1].hist(df['error'], bins=30); axes[1].axvline(0, color='r', ls='--')
    axes[1].set(title='Phân bố sai số (dự đoán − thật)', xlabel='người', ylabel='số ảnh')
    axes[2].bar(by_bin.index.astype(str), by_bin['MAE'])
    axes[2].set(title='MAE theo nhóm mật độ', xlabel='số người thật', ylabel='MAE')
    plt.suptitle(cfg['label']); plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR, f'test_mc_{key}_part{PART}.png'), bbox_inches='tight'); plt.show()

    # Ảnh minh họa bằng model của seed đầu tiên
    if not os.path.exists(recs[0]['weights']):
        print('Không có file trọng số nên bỏ qua phần ảnh minh họa.')
        return
    model = build_variant(key, pretrained=False)
    model.load_state_dict(torch.load(recs[0]['weights'], map_location=device))
    model.eval()
    ranked = df.assign(abs_err=df['error'].abs()).sort_values('abs_err')
    k = min(2, len(ranked))
    show_examples(model, key, ranked.head(k), f"{cfg['label']}: {k} ảnh test tốt nhất")
    show_examples(model, key, ranked.tail(k).iloc[::-1], f"{cfg['label']}: {k} ảnh test kém nhất")
    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

## 6. Train và kết quả

Train lần lượt từng seed (in tiến trình mỗi 25 epoch và mỗi lần có checkpoint tốt hơn), rồi tổng hợp kết quả.

In [ ]:
RESULTS['baseline'] = run_variant('baseline')

In [ ]:
report('baseline')

## 7. Tổng kết

Độ chính xác, kích thước và tốc độ. Bảng và hình được lưu vào `/kaggle/working`.

In [ ]:
s = MODEL_STATS['baseline']
summary = pd.DataFrame({'MobileCount (baseline)': {
    **{m: fmt_ms(*mean_std('baseline', m)) for m in ('MAE', 'MSE', 'GAME3', 'Đếm nhầm ở nền')},
    'MAE epoch cuối': fmt_ms(*mean_std('baseline', 'last_MAE')),
    'Tham số (triệu)': f"{s['params_M']:.2f}", 'GMACs': f"{s['GMACs']:.2f}" if s['GMACs'] else '–',
    'FPS fp32 / fp16': f"{s['FPS']:.1f} / {s['FPS_fp16']:.1f}"}}).T
summary.to_csv(os.path.join(WORK_DIR, f'baseline_mobilecount_summary_part{PART}.csv'))
print(f'Part {PART}, {len(test_recs)} ảnh test, {len(SEEDS)} seed. FPS trên {GPU_NAME}, ảnh {FPS_SIZE[1]}×{FPS_SIZE[0]}.')
display(summary)

per_seed = pd.DataFrame([{m: r['res'][m] for m in ('MAE', 'MSE')} for r in RESULTS['baseline']],
                        index=[f"seed {r['res']['seed']}" for r in RESULTS['baseline']])
per_seed.to_csv(os.path.join(WORK_DIR, f'baseline_mobilecount_seeds_part{PART}.csv'))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
x = np.arange(len(per_seed))
axes[0].bar(x - 0.2, per_seed['MAE'], 0.4, label='MAE')
axes[0].bar(x + 0.2, per_seed['MSE'], 0.4, label='MSE')
axes[0].set_xticks(x, per_seed.index)
axes[0].set(title='Kết quả test theo seed', ylabel='số người'); axes[0].legend(fontsize=8)
df = mean_preds('baseline')
by_bin = df.groupby(density_bin(df['gt']), observed=False)['error'].agg(MAE=lambda e: e.abs().mean(), lệch='mean')
by_bin.plot.bar(ax=axes[1], rot=0)
axes[1].set(title='MAE và độ lệch theo nhóm mật độ (TB các seed)', xlabel='số người thật')
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR, f'baseline_mobilecount_summary_part{PART}.png'), bbox_inches='tight')
plt.show()

if QUICK:
    print('\nĐÂY LÀ LẦN CHẠY THỬ (QUICK = True): số liệu không có ý nghĩa, không dùng cho báo cáo.')

print(f'\nTổng thời gian chạy notebook: {(time.time() - NOTEBOOK_START) / 3600:.2f} giờ. File kết quả trong {WORK_DIR}:')
for path in sorted(glob.glob(os.path.join(WORK_DIR, '*'))):
    if os.path.isfile(path):
        print(f'  {os.path.basename(path):45s} {os.path.getsize(path) / 1024:9.1f} KB')